In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" # Do not use GPU
import sys
from pathlib import Path
path = Path(os.getcwd())
parent_path = path.parent.absolute()
sys.path.append(os.getcwd())
sys.path.append(str(parent_path))
sys.path.append(os.path.join(os.getcwd(),'KnowledgeGraphEmbedding','codes'))
import logging
logger = logging.getLogger()
logger.basicConfig = logging.basicConfig(level=logging.DEBUG)
from data_generator import gen
from train import t_main
from predict_extract import check_MRR_Hits, get_best_logic_programs, check_soundness, check_pl, return_all_predicate, accuracy_all_relation
import argparse
from datetime import datetime
import shutil
import pickle
from pyDatalog import pyDatalog 
from tools.pre_data_gen import data_gen
from tools.lessthan_data_gen_pb import data_gen_lessthan
from tools.lessthan_data_gen_mis import data_gen_lessthan_mis
from sample import get_samle_dataset

ModuleNotFoundError: No module named 'data_generator'

In [8]:
from multiprocessing import set_start_method
import os
import sys

from pathlib import Path
import csv
import time
import random

from matplotlib.pyplot import show

#path = os.getcwczd()
path = Path(os.getcwd())
parent_path = path.parent.absolute()
sys.path.append(os.getcwd())
sys.path.append(str(parent_path))    
#sys.path.append(os.getcwd())
import logging
logger = logging.getLogger()
logger.basicConfig = logging.basicConfig(level=logging.DEBUG) # this is my first trial f
import itertools
import pickle 
import numpy as np

def classifier(raw_dataset_path, variable_number, original_data_path, t_relation = ''):
    '''
    Get some information about the relation, including the relation and objects, etc. 
    Comments: There is no random operations in this function. Run multiple times at different time from the same input will get the same output.
    '''
    all_rlation = {}
    save_all_relation = {}
    all_predicate = []
    all_object = {}
    relation = {}
    arity_relation = {}
    pro = {}
    with open(raw_dataset_path,'r') as f:
        new_single_line  = f.readline()        
        while new_single_line:
            one_perd = []
            # retrieval the relation name and two objects
            predicate = new_single_line[0:new_single_line.index(')')]
            if '#' in new_single_line:
                if 'TEST' in new_single_line:
                    new_single_line = f.readline()
                    continue
                if 'T.#' in new_single_line or 'T#' in new_single_line:
                    probability = 1
                else:
                    probability = new_single_line[new_single_line.index('.'):new_single_line.index('#')].replace('.','').replace(' ','').replace('#','')
                    probability = '0.'+probability
                    probability = float(probability) 
            else:
                probability = 1
            single_line = predicate.split('(')
            relation_name = single_line[0]
            the_rest = single_line[1].split(",")
            first_obj = the_rest[0]
            second_obj = the_rest[1]
            one_perd.append(relation_name)
            one_perd.append(first_obj)
            one_perd.append(second_obj)
            one_perd.append(probability)
            all_predicate.append(one_perd)
            if first_obj not in all_object:
                all_object[first_obj] = set([])
            if second_obj not in all_object:
                all_object[second_obj] = set([])
            if relation_name not in all_rlation:
                all_rlation[relation_name] = [set([]),set([])]
            relation[relation_name] = []
            arity_relation[relation_name] = 1 #  initial arity for all relation are 1
            new_single_line = f.readline()
            pro[predicate+')'] = probability
        f.close()
    all_relation_list = list(all_rlation)
    
    
    # Save all predicate name into the file 
    for i in all_relation_list:
        save_all_relation[i] = i
    with open(original_data_path+'/all_relation_dic.dt','wb') as f:
        pickle.dump(save_all_relation,f)
        f.close()
    with open(original_data_path+'/all_relation_dic.txt','w') as f:
        print(str(save_all_relation),file = f)
        f.close()  
    
    for pred in all_predicate:
        first_string = str(all_relation_list.index(pred[0])) + '-1'
        second_string = str(all_relation_list.index(pred[0])) + '-2'
        all_object[pred[1]].add(first_string)
        all_object[pred[2]].add(second_string)

    for pred in all_predicate:
        one_tuple = []
        one_tuple.append(pred[1])
        one_tuple.append(pred[2])
        one_tuple = tuple(one_tuple) 
        relation[pred[0]].append(one_tuple) # {'relation_name': (),(),...,()}

        # check the arity of all predicate 
        if pred[1] != pred[2] and arity_relation[pred[0]] == 1:
            arity_relation[pred[0]] = 2
    

    # Make variable - object dictionary
    variable_objects = {} # TODO this dictionary stores the classification of each variables in the task 
    for i in range(variable_number):
        variable_objects[i] = set([])
    
    target_arity = arity_relation[t_relation]
    if target_arity == 2:     
        for pred in all_predicate:
            if pred[0] == t_relation:
                variable_objects[0].add(pred[1])
                variable_objects[1].add(pred[2])
        # add variable into the rest of the variable 
        for obj in all_object:
            for variable_name in range(2,variable_number):
                variable_objects[variable_name].add(obj)
    elif target_arity == 1: # TODO. The arity of targetr predicate is 1. In order to make both postive and negative predicate, we ask all variables have the same classification 
        for obj in all_object:
            for variable_name in range(variable_number):
                variable_objects[variable_name].add(obj)
                # variable_objects[1].add(obj)
                # variable_objects[2].add(obj)
    
    # Save relation
    with open(original_data_path+'/relation_entities.dt','wb') as f:
        pickle.dump(relation, f)
        f.close()
    # Save the probability of all relational facts
    
                
    with open(original_data_path+'/pro.dt','wb') as f:
        pickle.dump(pro,f)
        f.close()
    with open(original_data_path+'/pro.txt', 'w') as f:
        print(pro,file=f)
        f.close()
    
    return variable_objects, list(relation.keys()), relation, arity_relation, target_arity

DEBUG:matplotlib:matplotlib data path: /Users/imansharifi/.pyenv/versions/3.12.8/lib/python3.12/site-packages/matplotlib/mpl-data
DEBUG:matplotlib:CONFIGDIR=/Users/imansharifi/.matplotlib
DEBUG:matplotlib:interactive is False
DEBUG:matplotlib:platform is darwin
DEBUG:matplotlib:CACHEDIR=/Users/imansharifi/.matplotlib
DEBUG:matplotlib.font_manager:Using fontManager instance from /Users/imansharifi/.matplotlib/fontlist-v390.json


In [9]:
raw_dataset_path = "test_dir/nations.nl"
var_objectives, relation_keys, relation, arity_relation, target_arity = classifier(raw_dataset_path, 2, 'test_dir', t_relation = 'blockpositionindex')

In [10]:
def get_all_valid_predicate_subsample(all_substitution, relation, all_variable_permination, t_relation, t_index, actual_n_substitutation, buffer_size, batch_size):
    '''
    The propositionalization method 2 mentioned by the paper:
    - Check whether there is a preidicate's value is 0 for always, and generated the corresponding valid body predicate. 
    - Different with the original function, this function adapts the subsampling process.  
    '''
    logging.info("The number of substitutation (circle in 'computed memory then stored in disk' manner):")
    logging.info(actual_n_substitutation)
    if actual_n_substitutation >= 0.2e7: # old threshold: 1.5e7
        return -1, -1
    flag_predicates = {}
    relation_index = 0
    relation_name = list(relation.keys())
    target_predicate_index = relation_name.index(t_relation)
    print("Target predicate is")
    print(target_predicate_index)
    for i in (relation):
        flag_predicates[relation_index] = [0] * len(all_variable_permination[i])
        relation_index += 1
    sub_index = 0
    start_time = time.time()

    finished_sub = 0
    initial_list = []
        
    # generate all trainable data in the formate of: x-y: [[S(x),S(y),S(z)]...][[1,1,0,0]^|number of predicates|... 
    while finished_sub < actual_n_substitutation:
        if finished_sub >= actual_n_substitutation:
            print(finished_sub, '/',actual_n_substitutation, 'Finish')
            break
        random_batch, initial_list = get_random_batch_elements(initial_list, all_substitution, batch_size, buffer_size)
        # Doing substitutions
        for i in random_batch:
            current_relation_index = 0
            for relation_name in relation:  
                current_data = relation[relation_name]
                current_predicate_index = 0
                current_permination = all_variable_permination[relation_name]
                for j in current_permination:
                    first_variable = j[0]
                    second_variable = j[1]
                    target_tuple = []
                    target_tuple.append(i[first_variable])
                    target_tuple.append(i[second_variable])
                    target_tuple = tuple(target_tuple)
                    if target_tuple in current_data:
                        flag_predicates[current_relation_index][current_predicate_index] = 1
                    current_predicate_index += 1
                current_relation_index += 1
            
            sub_index+=1
            if sub_index % 500 == 0:
                start_time = show_process(500, actual_n_substitutation, sub_index, start_time, note='[Valid Features with Percent]')
        finished_sub += batch_size

    print("All predicate info is")
    print(flag_predicates)
    valid_index = []
    number_of_predicate = 0
    for i in flag_predicates:
        info = flag_predicates[i]
        for j in info:
            if j == 1 and number_of_predicate != t_index:
                valid_index.append(number_of_predicate)
            number_of_predicate += 1
            
    # Compute whether all boolean variable in the body are zero
    logging.info("Valid predicare index are")
    logging.info(valid_index)
    template = {}
    number_of_predicate = 0
    for i in flag_predicates:
        template[i] = []
        for j in range(len(flag_predicates[i])):
            if flag_predicates[i][j] == 1 and number_of_predicate != t_index:
                template[i].append(j)
            number_of_predicate += 1 
            
    logging.info("Valid template")
    logging.info(template)
    logging.info("Check Valid Predicate Success")
    return valid_index, template

def get_all_valide_predicate(all_substitution, relation, all_variable_permination, t_relation, t_index, number_all_sub = 0):
    '''
    The propositionalization method 2 mentioned by the paper:
    - Check whether there is a preidicate's value is 0 for always, and generated the corresponding valid body predicate. 
    '''
    logging.info('The total number of all substitutions is (compute from memory then disk at once time.):')
    logging.info(number_all_sub)
    if number_all_sub >= 1.5e7:
        return -1, -1
    flag_predicates = {}
    relation_index = 0
    relation_name = list(relation.keys())
    target_predicate_index = relation_name.index(t_relation)
    print("Target predicate is")
    print(target_predicate_index)
    for i in (relation):
        flag_predicates[relation_index] = [0] * len(all_variable_permination[i])
        relation_index += 1
    sub_index = 0
    
    start_time = time.time()
    # generate all trainable data in the formate of: x-y: [[S(x),S(y),S(z)]...][[1,1,0,0]^|number of predicates|...]
    for i in all_substitution:  
        current_relation_index = 0
        for relation_name in relation: #! fix at here 
            current_data = relation[relation_name]
            current_predicate_index = 0
            current_permination = all_variable_permination[relation_name]
            for j in current_permination:
                first_variable = j[0]
                second_variable = j[1]
                target_tuple = []
                target_tuple.append(i[first_variable])
                target_tuple.append(i[second_variable])
                target_tuple = tuple(target_tuple)
                if target_tuple in current_data:
                    flag_predicates[current_relation_index][current_predicate_index] = 1
                current_predicate_index += 1
            current_relation_index += 1

        
        sub_index+=1
        if sub_index % 2000 == 0:
            start_time = show_process(2000, number_all_sub, sub_index,start_time, note='[Find Valid Features]')
            # print(sub_index, flush=True)   
    
    print("All predicate info is")
    print(flag_predicates)
    valid_index = []
    number_of_predicate = 0
    for i in flag_predicates:
        info = flag_predicates[i]
        for j in info:
            if j == 1 and number_of_predicate != t_index:
                valid_index.append(number_of_predicate)
            number_of_predicate += 1
            
    # Compute whether all boolean variable in the body are zero
    print("Valid predicare index are")
    print(valid_index)
    template = {}
    number_of_predicate = 0
    for i in flag_predicates:
        template[i] = []
        for j in range(len(flag_predicates[i])):
            if flag_predicates[i][j] == 1 and number_of_predicate != t_index:
                template[i].append(j)
            number_of_predicate += 1 
            
    print("Valid template")
    print(template)
    return valid_index, template

def show_process(step, all_process, current_process, start_time, note = ''):
    '''
    Print the process and the running time and the left time.
    Parameters: 
    step: the interval to print info 
    all_process: all number of substitutions
    current_process: current number substitution
    start_time: the initial start time.
    '''
    process = float(current_process*100/all_process)
    step_time = float((time.time() - start_time)/60.0)
    rest_running_time = float(((all_process - current_process) / step) * step_time)
    if rest_running_time > 60:
        print(note + ' Percent: %.2f'%process,'%','Current step: %d'%int(current_process),'Total steps: %d'%int(all_process),'Remain running time: %.2f (hours).'%float(rest_running_time/60), end='\r')
    else:
        print(note + ' Percent: %.2f'%process,'%','Current step: %d'%int(current_process),'Total steps: %d'%int(all_process),'Remain running time: %.2f (minutes).'%rest_running_time, end='\r')
    return time.time()
    
    
    
def make_all_predicate(relation, relation_variable_permutations):
    '''
    Generate all first-order features in the dataset.
    '''
    a_p_a_v = []
    for relation_name in relation: #
        current_permination = relation_variable_permutations[relation_name]
        for variable_pair in current_permination:
            a_p_a_v.append(variable_pair)
    return a_p_a_v

def make_small_data(all_substitution, relation, relation_variable_permutations, template, pro, t_relation, train_label,original_data_path, actual_n_substitutation):
    '''
    Generate trainable data when datasets are samll.
    - ALL substitutions based on the all objects;
    - Generate all trainable data in the formate of: x-y: [[S(x),S(y),S(z)]...][[1,1,0,0]^|number of predicates|...]
    '''
    times_substitution = 0
    valid_substitution = 0
    start_time = time.time()
    for single_substitution in all_substitution: 
        relation_index = 0
        # The first constrain mentioned in the paper: if all of the input are 0, then the labels should not be one 
        wrong_flag = True  
        
        all_body_boolean_value = {} # used to record all boolean values of the body of the logic program
            
        for relation_name in relation: #
            y_one_data = []
            data = relation[relation_name] # Read all relational data 
            current_permination = relation_variable_permutations[relation_name]
            for variable_pair in current_permination:
                string_tuple = []
                # begin to save variable arrgement 
                for m in variable_pair:
                    string_tuple.append(single_substitution[m])
                string_tuple = tuple(string_tuple) # like (sue, dinana)...
                if string_tuple in data:
                    # build the current symbolic relational data 
                    current_fact = relation_name+'('+string_tuple[0] +',' + string_tuple[1]+')'
                    prob_value = pro[current_fact]
                    y_one_data.append(prob_value)
                else:
                    y_one_data.append(0)
            
            all_body_boolean_value[relation_index]=y_one_data
            relation_index += 1   
        times_substitution += 1

        # Compute whether all boolean variable in the body are zero
        check_ind = 0
        for i in all_body_boolean_value:
            label_list=[]
            for acq in template[check_ind]:     #Check the tempalte data
                label_list.append( all_body_boolean_value[i][acq])
            if 1 in label_list:
                # ! change to false to check whether accuracy of neural predicate are improved
                wrong_flag = False 
                break
            check_ind += 1
        # target predicate is ancester(x,y) -> ancester(0,1) in the embedding point of view 
        target_tuple = []
        # append the subtitution value corresponding to the variable x
        target_tuple.append(single_substitution[0]) 
        # append the subtitution value corresponding to the variable y
        target_tuple.append(single_substitution[1]) 
        # The symbolic format data correspinding to the variables x and y
        target_tuple = tuple(target_tuple) 

        if target_tuple in relation[t_relation] and wrong_flag == True:
            continue
        
        # TODO STep 1: Add all non-target predicate into train dataset  
        for i in all_body_boolean_value:
            train_label[i].append(all_body_boolean_value[i])

        # TODO Step 2: Add the target predicare firsr-order feature into the train data
        if target_tuple in relation[t_relation]:
            current_fact = t_relation+'('+target_tuple[0] +',' + target_tuple[1]+')'
            current_fact_pro = pro[current_fact]
            train_label[relation_index].append([current_fact_pro])
        else:
            train_label[relation_index].append([0])            
        valid_substitution += 1
        if times_substitution % 5000 == 0:
            start_time = show_process(5000, actual_n_substitutation, times_substitution, start_time, note='[Make Trainable data]')
            
    train_label[len(relation)+1] = train_label[len(relation)]
    
    print("👉 The generated data is:")
    print(len(train_label), len(train_label[0]), len(train_label[1]), len(train_label[2]) )

    final_x = []
    for i in range(len(train_label) - 1):
        final_x.append(train_label[i])

    new_train_label = train_label[len(train_label) - 1]

    x_file = original_data_path + 'x_train.dt' 
    y_file = original_data_path + 'y_train.dt' 


    with open(x_file, 'wb') as xf:
        pickle.dump(final_x,xf)
        xf.close()
    with open(y_file, 'wb') as yf:
        pickle.dump(new_train_label,yf)
        yf.close()
    return valid_substitution, times_substitution

    
    
def make_large_data(all_substitution, relation, relation_variable_permutations, template, pro, t_relation,original_data_path, actual_n_substitutation, batch_size, buffer_size):
    '''
    Generate large datasets
    - ALL substitutions based on the all objects;
    - Generate all trainable data in the formate of: x-y: [[S(x),S(y),S(z)]...][[1,1,0,0]^|number of predicates|...]
    '''
    datasets_path = original_data_path + t_relation+'/data.tfr' 
    writer = tf.io.TFRecordWriter(datasets_path)
    times_substitution = 0 
    valid_substitution = 0
    start_time = time.time()
    initial_list = []
    while times_substitution < actual_n_substitutation:
        batch_substitution, initial_list = get_random_batch_elements(initial_list, all_substitution, batch_size, buffer_size)
        for single_substitution in batch_substitution: 
            times_substitution += 1
            relation_index = 0
            # The first constrain mentioned in the paper: if all of the input are 0, then the labels should not be one 
            wrong_flag = True  
            
            all_body_boolean_value = {} # used to record all boolean values of the body of the logic program
                
            for relation_name in relation: #
                y_one_data = []
                data = relation[relation_name] # Read all relational data 
                current_permination = relation_variable_permutations[relation_name]
                for variable_pair in current_permination:
                    string_tuple = []
                    for m in variable_pair:
                        string_tuple.append(single_substitution[m])
                    string_tuple = tuple(string_tuple) # like (sue, dinana)...
                    if string_tuple in data:
                        # build the current symbolic relational data 
                        current_fact = relation_name+'('+string_tuple[0] +',' + string_tuple[1]+')'
                        prob_value = pro[current_fact]
                        y_one_data.append(prob_value)
                    else:
                        y_one_data.append(0)
                
                all_body_boolean_value[relation_index]=y_one_data
                relation_index += 1   
            

            # Compute whether all boolean variable in the body are zero
            check_ind = 0
            for i in all_body_boolean_value:
                label_list=[]
                for acq in template[check_ind]:     #Check the tempalte data
                    label_list.append( all_body_boolean_value[i][acq])
                if 1 in label_list:
                    # ! change to false to check whether accuracy of neural predicate are improved
                    wrong_flag = False 
                    break
                check_ind += 1
            # target predicate is ancester(x,y) -> ancester(0,1) in the embedding point of view 
            target_tuple = []
            # append the subtitution value corresponding to the variable x
            target_tuple.append(single_substitution[0]) 
            # append the subtitution value corresponding to the variable y
            target_tuple.append(single_substitution[1]) 
            # The symbolic format data correspinding to the variables x and y
            target_tuple = tuple(target_tuple) 

            if target_tuple in relation[t_relation] and wrong_flag == True:
                continue
            
            body = []
            head = []
            
            # TODO STep 1: Add all non-target predicate into train dataset  
            for i in all_body_boolean_value:
                for j in all_body_boolean_value[i]:
                    body.append(np.float64(j))
            
            # TODO Step 2: Add the target predicare firsr-order feature into the train data
            if target_tuple in relation[t_relation]:
                current_fact = t_relation+'('+target_tuple[0] +',' + target_tuple[1]+')'
                current_fact_pro = pro[current_fact]
                head.append([np.float64(current_fact_pro)])
                body.append(np.float64(current_fact_pro))
            else:
                head.append([np.float64(0)])
                body.append(np.float64(0))
            
            tf_example = first_order_features_example(body,head)
            writer.write(tf_example.SerializeToString())
                
            valid_substitution += 1
            if times_substitution % 5000 == 0:
                start_time = show_process(5000, actual_n_substitutation, times_substitution, start_time, note='[Make trainable data in Large mode]')
            
    writer.close()
    return valid_substitution, times_substitution



In [11]:
def main(dataset_name, t_relation = '', path_name = '' , original_data_path='', variable_depth=1, large = False, sub_per = 0.02, buffer_size = 9000, random_size = 5000):

    relation_name = []
    variable_number = variable_depth + 2 
    variable_objects, relation_name, relation , arity_relation, target_arity = classifier(original_data_path+dataset_name, variable_number, original_data_path, t_relation)
    logging.info('Begin generating data with %s as head predicate.'%t_relation)

    # Assemb all the countries 
    variable_class = {}     # cityLocIn(x,y)  x->countries y->region z->subregion, cities, and regions. 
    # Then possible predicate include: locatedIn[c_2,c_2],locatedIn[c_1,c_2], locatedIn[c_1,c_3], locatedIn[c_2,c_3] neighbor[c_1,c_1]
    for i in variable_objects:
        variable_class[i] = list(variable_objects[i])
        
    ALL_objects = []
    for i in variable_class:
        l = variable_class[i]
        for j in l:
            if j not in ALL_objects:
                ALL_objects.append(j)

        
    entities_dic_name_number = {}  #sue:0 
    
    for i in ALL_objects:
        entities_dic_name_number[i] = ALL_objects.index(i) 
    print('👉 The second dictionary is', entities_dic_name_number)

    print("👉 All relation in the dataset")
    print(relation)
    
    # train the neural predicates
    variable_value = variable_class.values()
    # Compute the number of all substitutation
    all_substitution_number = 1
    for i in variable_value:
        all_substitution_number *= len(i)
    if large == False:
        all_substitution = list(itertools.product(*variable_value))
    else:
        all_substitution = itertools.product(*variable_value)
    
    if large == True:
        actual_n_substitutation = int(all_substitution_number * sub_per)
    else:
        actual_n_substitutation = all_substitution_number
    
    
    # The possible predicate correspodnnig each predicates 
    relation_variable_permutations = {}
    for i in arity_relation:
        if arity_relation[i] == 1:
            empty_list = []
            for variable_index in range(variable_number):
                empty_list.append((variable_index, variable_index))
            relation_variable_permutations[i] = empty_list
        else:
            relation_variable_permutations[i] = list(itertools.permutations(range(variable_number), 2 ))

    # The index of the target predicate 
    find_index = 0
    for i in range(len(relation_name)):
        for j in relation_variable_permutations[relation_name[i]]:
            if arity_relation[t_relation] == 1 and j == (0,0) and i == relation_name.index(t_relation):
                t_index = find_index
                target_predicate = t_relation+"(X,X)"
            elif arity_relation[t_relation] == 2 and j == (0,1) and i == relation_name.index(t_relation):
                t_index = find_index
                target_predicate = t_relation+"(X,Y)"
            find_index += 1
                
    print('The index of target predicate are:')
    print(t_index)

    print('👉 all variables')
    print(relation_variable_permutations)
    with open(path_name+'relation_variable.dt','wb') as f:
        pickle.dump(relation_variable_permutations, f)
        f.close()
    with open(path_name+'relation_variable.txt','w') as f:
        print(relation_variable_permutations, file=f)
        f.close()    
    

    train_label = {}
    for i in range(len(relation)):
        train_label[i] = []

    # Used to store the Boolean label for the taget relational predicate
    target_predicate_index = len(relation)  
    train_label[target_predicate_index] = []
        
    print("👉 Pre-operation on the label dataset:", train_label)
    

    try:
        with open(path_name + 'valid_index.dt', "rb") as f:
            res = pickle.load(f)
            valid_index = res['valid_index']
            template = res['template']
            print(res)
            f.close()
    except FileNotFoundError:
        if large == True:
            valid_index, template = get_all_valid_predicate_subsample(all_substitution, relation, relation_variable_permutations, t_relation, t_index, actual_n_substitutation, buffer_size, random_size)
        else:
            valid_index, template= get_all_valide_predicate(all_substitution=all_substitution,relation = relation, all_variable_permination=relation_variable_permutations, t_relation = t_relation, t_index = t_index, number_all_sub=actual_n_substitutation )
        
        if valid_index == -1 and template == -1:
            return -1,-1,-1
        res = {}
        res['valid_index'] = valid_index
        res['template'] = template
        with open(path_name + 'valid_index.dt', 'wb') as f:
            pickle.dump(res, f)
            f.close()
        with open(path_name + 'valid_index.txt','w') as f:
            print(str(res), file = f)
            f.close()
        print("Save template succeess")

    if large == True:
        all_substitution = itertools.product(*variable_value)

    # open the probabilistic table 
    with open(path_name+'pro.dt',"rb") as f:
        pro = pickle.load(f)
        f.close()
    
    # Make trainable data
    if large == True: 
        valid_substitution, times_substitution =  make_large_data(all_substitution, relation, relation_variable_permutations, template, pro, t_relation, original_data_path, actual_n_substitutation, random_size, buffer_size)
    else:
        valid_substitution, times_substitution = make_small_data(all_substitution, relation, relation_variable_permutations, template, pro, t_relation, train_label, original_data_path, actual_n_substitutation)

    # Print some meta-information into datasets
    with open(original_data_path + 'number_valid_substitution.txt', 'w') as vf:
        vf.write(str(valid_substitution))
        logging.info(str(valid_substitution)+' (valid)/(all) '+str(times_substitution))
        vf.close()
    
    # Save all variable permutations into the corresponding file
    print("👉 Begin to save all variable permutations in all predicates...")
    a_p_a_v = make_all_predicate(relation, relation_variable_permutations)
    with open(original_data_path + 'predicate.txt', 'w') as vf:
        vf.write(str(a_p_a_v))
        vf.close()
    with open(original_data_path + 'predicate.list', 'wb') as vf:
        pickle.dump((a_p_a_v), vf)
        vf.close()
    print("👉 Save success")
    
    
    all_obj_file = original_data_path + 'all_ent.dt'
    with open(all_obj_file, 'wb') as f:
        pickle.dump(ALL_objects,f)
        f.close()    
        
    # return all the objects in the task 
    return ALL_objects , target_arity, target_predicate


def gen(dataset_name, target_relation_name, variable_depth, large, sub_per, buffer_size, random_size):
    original_data_path = 'test_dir/'
    path_name = 'test_dir/'
    _, target_arity, head_pre = main(dataset_name, t_relation=target_relation_name, path_name=path_name,original_data_path=original_data_path, variable_depth=variable_depth, large = large, sub_per=sub_per, buffer_size=buffer_size, random_size=random_size)
    if target_arity == -1 and head_pre == -1:
        return -1, -1
    print("-------Generating Data Success!-------")
    return target_arity, head_pre

In [12]:
def data_assemble(dataset_name, predicate, variable_depth, large = False, sub_per = 0.2, buffer_size = 9000, random_size = 5000):
    target_arity, head_pre = gen(dataset_name, predicate, variable_depth, large, sub_per, buffer_size = buffer_size, random_size = random_size)
    if target_arity == -1 and head_pre == -1:
        logging.warning("Data generation on checking valid predicate time is too long!")
        raise NameError('RUNTOOLONG')
    meta_info = {'target_arity':target_arity, "head_pre": head_pre}
    with open("test_dir/meta_info.dt",'wb') as f:
        pickle.dump(meta_info,f)
        f.close()
    with open("test_dir/meta_info.txt",'w') as f:
        print(meta_info, file=f)
        f.close()
    logging.info("Generate all trainable data success with variable depth:")
    logging.info(variable_depth)
    return 0 

In [13]:
data_assemble('even.nl', 'even', 1, large = False, sub_per = 0.2, buffer_size = 9000, random_size = 5000)

IndexError: list index out of range

In [1]:
import pickle
# read the x_train.dt and y_train.dt files
dataset_dir = 'datasets/classics/connectedness/connectedness/'
with open(dataset_dir + 'x_train.dt', 'rb') as xf:
    x_train = pickle.load(xf)
    xf.close()
with open(dataset_dir + 'y_train.dt', 'rb') as yf:
    y_train = pickle.load(yf)
    yf.close()

print("👉 The x_train data is:")
print(len(x_train[0]))
print("👉 The y_train data is:")
print(len(y_train))

👉 The x_train data is:
192
👉 The y_train data is:
192


In [2]:
len(x_train)

3

In [3]:
import numpy as np
x_train_np = np.concatenate(x_train, axis=1)
x_train_np.shape

(192, 25)

In [6]:
x_train_np[5]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [7]:
y_train[5]

[0]

In [10]:
# count the number of positive samples in y_train
positive_count = 0
for i in y_train:
    if i[0] > 0:
        positive_count += 1
print("Number of positive samples in y_train:", positive_count)

Number of positive samples in y_train: 10192


In [4]:
import pickle

# Load the predicate list
with open(dataset_dir + 'predicate.list', 'rb') as f:
    predicate_list = pickle.load(f)
    f.close()

# Load relation variable permutations
with open(dataset_dir + 'relation_variable.dt', 'rb') as f:
    relation_variable_permutations = pickle.load(f)
    f.close()

# Load relation names
with open(dataset_dir + 'all_relation_dic.dt', 'rb') as f:
    all_relations = pickle.load(f)
    f.close()

print("Predicates:", len(predicate_list))
print("Relations:", len(all_relations))
print("Variable permutations:", len(relation_variable_permutations))

Predicates: 24
Relations: 2
Variable permutations: 2


In [9]:
print("First 10 predicates:", predicate_list[:10])

First 10 predicates: [(0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 3), (3, 0)]


In [5]:
def get_predicate_name(index, predicate_list, relation_variable_permutations, all_relations):
    """
    Convert a feature index in X_train to its symbolic predicate name.
    """
    # Get variable mapping (e.g., (0, 1) for X, Y)
    var_tuple = predicate_list[index]
    
    # Variable name mapping
    var_names = {0: 'X1', 1: 'X2', 2: 'X3', 3: 'X4'}
    
    # Find which relation this belongs to
    cumulative_index = 0
    for relation_name, permutations in relation_variable_permutations.items():
        if cumulative_index <= index < cumulative_index + len(permutations):
            # Found the relation
            local_index = index - cumulative_index
            var_pair = permutations[local_index]
            
            # Build symbolic predicate
            symbolic = f"{relation_name}({var_names[var_pair[0]]}, {var_names[var_pair[1]]})"
            return symbolic
        
        cumulative_index += len(permutations)
    
    return "Unknown"

# Example: Map X_train column 0
for idx in range(len(predicate_list)):
    predicate_name = get_predicate_name(idx, predicate_list, 
                                        relation_variable_permutations, 
                                        all_relations)
    print(f"X_train[:, {idx}] = {predicate_name}")

X_train[:, 0] = edge(X1, X2)
X_train[:, 1] = edge(X1, X3)
X_train[:, 2] = edge(X1, X4)
X_train[:, 3] = edge(X2, X1)
X_train[:, 4] = edge(X2, X3)
X_train[:, 5] = edge(X2, X4)
X_train[:, 6] = edge(X3, X1)
X_train[:, 7] = edge(X3, X2)
X_train[:, 8] = edge(X3, X4)
X_train[:, 9] = edge(X4, X1)
X_train[:, 10] = edge(X4, X2)
X_train[:, 11] = edge(X4, X3)
X_train[:, 12] = connectedness(X1, X2)
X_train[:, 13] = connectedness(X1, X3)
X_train[:, 14] = connectedness(X1, X4)
X_train[:, 15] = connectedness(X2, X1)
X_train[:, 16] = connectedness(X2, X3)
X_train[:, 17] = connectedness(X2, X4)
X_train[:, 18] = connectedness(X3, X1)
X_train[:, 19] = connectedness(X3, X2)
X_train[:, 20] = connectedness(X3, X4)
X_train[:, 21] = connectedness(X4, X1)
X_train[:, 22] = connectedness(X4, X2)
X_train[:, 23] = connectedness(X4, X3)


In [20]:
# check if y_train is equal to column 252 in X_train
target_column_index = 252
mismatch_count = 0
for i in range(len(y_train)):
    if y_train[i][0] != x_train_np[i][target_column_index]:
        mismatch_count += 1
print("Number of mismatches between y_train and X_train column 252:", mismatch_count)

Number of mismatches between y_train and X_train column 252: 0


In [ ]:
# Load x_train (list of arrays, one per relation)
with open(x_file, 'rb') as xf:
    x = pickle.load(xf)  # x is a list: [relation_0_data, relation_1_data, ...]
    xf.close()

# Combine all predicates from all relations into ONE vector
x = np.concatenate(x, axis=1)  # Concatenate along columns
# Result: shape [num_samples, total_predicates]

number_all_predicate = int(np.shape(x)[1])  # Total number of features

In [ ]:
import itertools
import tensorflow as tf 
import tensorflow.keras as keras
from tensorflow.python.keras.backend import shape
from tensorflow.python.ops.gen_batch_ops import batch

import pickle
import numpy as np

class FuzzyOrLayer(keras.layers.Layer):
    def call(self, inputs):
        neg_inputs = 1 - inputs
        pro_neg_input = tf.math.reduce_prod(neg_inputs, 1)
        predict_value = 1 - pro_neg_input
        return tf.reshape(predict_value, [-1, 1])

#TODO: 1. Delete all codes relatived with the original version which not includs target_index 
# Define the layer of D-LFIT:
# The input of all layers is a vector embedding all predicates from all relations in tasks. 
class inference(keras.layers.Layer):     # unit is the default number of different logic programs for single relation predicate  
    '''
    - self.target_index is a integer to show th location of the target predicates
    - self.templtae show the tuple of the templates ! need to change to template_predi_index, the latter one
    is more accurate
    '''
    def __init__(self, n_rules, alpha, data_path, path, 
                 template, target_index, arity, l_name, 
                 prior_knowledge=False, variable_number=3):
        self.rules = n_rules
        self.alpha = alpha
        self.path_ = path 
        # self.template = template
        self.template_predi_index = template
        self.template_predi = []
        self.target_index = target_index 
        self.arity = arity
        self.prior_knowledge = prior_knowledge 
        self.variable_number = variable_number
        if prior_knowledge != False:
            with open(data_path + 'prior_knowledge.dt', 'rb') as f:
                self.prior_knowledge = pickle.load(f)
                f.close()
            self.prior_knowledge_number = np.shape(self.prior_knowledge)[1]
            
        super(inference, self).__init__(name = l_name)
        
    # Define the customized sigmoid-like activation function
    def sigmoid_like(self, x):
        y = tf.math.divide(1,(1+tf.math.exp(-1 * self.alpha *(x))))  
        return y
    
    def build(self, input_shape): 
        # Make embeddings for all predicates in the output of the neural predicate layers
        with open(self.path_,'rb') as vf:
            all_predicates = pickle.load(vf)
            vf.close()
        print("[⭐️ np] All predicates in the task are:", all_predicates)

        # if self.template_predi_index != None:
        # TODO Load the valid index of predicates
        for i in self.template_predi_index:
            self.template_predi.append(all_predicates[i])
        print("⭐️ The indexes are:")
        print(self.template_predi_index)
        print("⭐️ Selected predicate according to the template are")
        print(self.template_predi)
        
        if type(self.prior_knowledge) is not bool:
            # This operation just increaset the loss value from the begining stage 
            # ! we plan to set this weight value very large 
            self.prior_weights = tf.Variable(initial_value=self.prior_knowledge, trainable=False, name = 'prior') 
            self.w = tf.Variable(
                initial_value=tf.random.truncated_normal(shape=(len(self.template_predi_index), self.rules), dtype="float32"),
                trainable=True,
                constraint=lambda t: tf.clip_by_value(t, 0, 1), name = 'rules_weights')
        else:
            self.w = tf.Variable(
                initial_value=tf.random.truncated_normal(shape=(len(self.template_predi_index), self.rules), dtype="float32"),
                trainable=True,
                constraint=lambda t: tf.clip_by_value(t, 0, 1), name='rules_weights'
            )
        # else: 
        #     self.w = tf.Variable(
        #         initial_value=tf.random.truncated_normal(shape=(input_shape[-1]-1, self.rules), dtype="float32"), # The node expecting the target predicates
        #         trainable=True,
        #         constraint=lambda t: tf.clip_by_value(t, 0, 1)
        #     )
        # TODO Perpare for the embeedding for the template predicates 
        embedddings_o_p = []
        if self.template_predi_index == None:
            valid_pred = all_predicates
        else:
            valid_pred = self.template_predi
        
        for i in range(len(valid_pred)):
            item = valid_pred[i]
            one_label = []
            # make basic and occurrence embeddings for each valid first-order features
            for variable_index in range(self.variable_number):
                if item[0] == variable_index or item[1] == variable_index: # Check whether variable x is in the tuple item (0,2) <-> (x,z)
                    one_label.append(1)
                else:
                    one_label.append(0)
            # if item[0] == 0 or item[1] == 0: # Check whether variable x is in the tuple item (0,2) <-> (x,z)
            #     one_label.append(1)
            # else:
            #     one_label.append(0)
            # if item[0] == 1 or item[1] == 1: # Check whether variable y is in the tuplte item 
            #     one_label.append(1)
            # else:
            #     one_label.append(0)
            # if item[0] == 2 or item[1] == 2: # Check whether variable z is in the tuplte item 
            #     one_label.append(1)
            # else:
            #     one_label.append(0)
            embedddings_o_p.append(one_label)
        print("[⭐️ np] All emnbeddings for output predicate are:")   
        print(embedddings_o_p)
        self.e_o_p = tf.concat(np.array(embedddings_o_p, dtype=np.float32), 0)
        self.e_o_p = tf.Variable(initial_value=self.e_o_p, trainable=False)
        print("[⭐️ np] The shape of the embeddings of all predicates after concation:")
        print(self.e_o_p)
        self.e_o_p = tf.reshape(self.e_o_p, [self.variable_number,-1])
        print("[⭐️ np] The shape of the embeddings of all predicates after transpose:")
        print(self.e_o_p)
        super().build(input_shape)
        return 0 
          
    def sptial_activation(self, x):
        a = 1
        b = 10
        c = 1
        y =  c * tf.math.exp(a-b*tf.math.square(x-1))
        return y
            
    def fuzzy_or(self, inputs):
        neg_inputs = 1 - inputs
        pro_neg_input = tf.math.reduce_prod(neg_inputs, 1)
        predict_value = 1 - pro_neg_input
        predict_value = tf.reshape(predict_value, [-1, 1])
        return predict_value
        
    def spatial_unity(self, inputs):
        '''
        Define the sptial unity:
        - 1. Ensure that the variable x and y appear in the body
        - 2. Ensure the occurrence of variable z in the body cannnot be one 
        '''
        # Do multiple between parameters and its embeddings. The first arity         
        weights = self.w 
        weights = tf.transpose(weights)
        weights = tf.expand_dims(weights,1)
        weighted_embeddings = weights * self.e_o_p
        constrain_x_and_y = weighted_embeddings[:,0:self.arity,:] # keep the first arity embeddings 
        constrain_rest = weighted_embeddings[:,self.arity:,:] # The latter embeddigns 
        neg_weighted_embedddings = 1-constrain_x_and_y
        reduce_pro = tf.math.reduce_prod(neg_weighted_embedddings, 2)
        fuzzy_or =  1 - reduce_pro # This result is the fuzzy or between all predicate for same boolean variable and
        print(fuzzy_or)
        fuzzy_and = tf.math.reduce_prod(fuzzy_or, 1) # This result is the fuzzy and between all Boolean variable in the all rules 
        print(fuzzy_and)
        fuzzy_and = tf.reshape(fuzzy_and, [-1, self.rules])
        # all_rule_fuzzy_and = tf.math.reduce_prod(fuzzy_and, 0)
        # all_rule_fuzzy_and = tf.reshape(all_rule_fuzzy_and, [-1,1])
        print("[⭐️ np]: The final of sptial output")
        print(fuzzy_and)
        sptial_output = tf.matmul(inputs, fuzzy_and, name='sptial_utility') 

        # The second arity 
        the_sum_all_rest = tf.math.reduce_sum(constrain_rest,2)
        sptial_activate = self.sptial_activation(the_sum_all_rest)
        sum_sptial_activate = tf.math.reduce_sum(sptial_activate, 1)
        sptial_activate_transpose = tf.reshape(sum_sptial_activate, [-1, self.rules])
        sptial_output_2 = tf.matmul(inputs, sptial_activate_transpose, name='sptial_utility_rest_variable')
        
        return sptial_output, sptial_output_2
        
    def target_predicate_predited_neural(self,inputs): # predicted by the neural predicates
        print(inputs)
        target_predicate_predicted = tf.gather(inputs, self.target_index, axis = 1, name='direct_predi')
        target_predicate_predicted=tf.reshape(target_predicate_predicted, [-1,1],name='direct_predi')
        return target_predicate_predicted
    
    def call(self, inputs,fun_index):  #! Upate at here when logic templates are used.
        logic_program_layer = {}
        if fun_index == 1:
            # if self.template_predi_index == None:
            #     if self.target_index == None:
            #         t_head = inputs[:,:self.target_index]
            #         t_tail = inputs[:,self.target_index+1:]
            #         inputs = tf.concat([t_head, t_tail], axis = 1)
            #         logic_program_layer['inference_output'] = self.sigmoid_like(tf.matmul(inputs,self.w,name='inference_with_tem')-1)
            #     else:
            #         logic_program_layer['inference_output'] = self.sigmoid_like( tf.matmul(inputs, self.w,name='infer') - 1 ) # ! change at here
            # else:
            template_predicate = tf.gather(inputs, self.template_predi_index, axis=1)
            print("⭐️ The original output of neural predicate are:")
            print(inputs)
            print("⭐️ The cutten output after logic tem[late are")
            print(template_predicate)
            logic_program_layer["inference_output"] = self.sigmoid_like( tf.matmul(template_predicate, self.w) -1)
        if fun_index == 2:
            '''
            Make the sum of all weights in a single rule be one. 
            '''
            sum_tem = tf.reduce_sum(self.w, 0)
            sum_tem = tf.reshape(sum_tem, [1,-1])
            print(sum_tem.shape)
            logic_program_layer['one_sum_loss'] = tf.matmul(inputs, sum_tem, name='one_sum')    
            print("[⭐️ np]: Shape of one_sum",logic_program_layer['one_sum_loss'] )     

        if fun_index == 3:
            all_combination_rows = list(itertools.product(range(self.prior_knowledge_number), range(self.rules)))          
            cosine_loss = tf.keras.losses.CosineSimilarity(axis=1)
            dissimi = []
            for j in all_combination_rows:
                y_1 = tf.reshape(self.prior_weights[:,j[0]], [1,-1])
                y_2 = tf.reshape(self.w[:,j[1]], [1,-1])
                diss = tf.reshape(tf.convert_to_tensor(cosine_loss(y_1, y_2)), [1])
                dissimi.append(diss)
            dissmi_tf = tf.concat(dissimi,axis=0)
            # y_1 = all_combination_rows[:,0]
            # y_2 = all_combination_rows[:,1]
            # cosine_loss = tf.keras.losses.CosineSimilarity(axis=1)
            dissimilarity = tf.reshape( dissmi_tf,[1,-1])
            final_dis = tf.matmul(inputs, dissimilarity)
            logic_program_layer['cos_prior'] = final_dis # To MSE (cos_prior-0)^2
        return logic_program_layer
    
    def get_config(self):
        return {"inference_weights": self.w.numpy(), "sptial_embeddings":self.e_o_p.numpy()}

    @classmethod
    def from_config(cls, config):
        return cls(**config)

    def return_template_predi_index(self):
        return self.template_predi_index
           
class auxiliary_inference(inference):
    def __init__(self, addtional_aux_num, aux_prior_knowledge, **kwds):
        self.addtional_aux_num = addtional_aux_num
        
        super().__init__(**kwds)
        
    
    def build(self,input_shape):
        super().build(input_shape=input_shape)
        # Make embeddings for all predicates in the output of the neural predicate layers
        with open(self.path,'rb') as vf:
            all_predicates = pickle.load(vf)
            vf.close()
        print("[⭐️ np] All predicates in the task are:", all_predicates)

        if self.template_predi_index != None:
            for i in self.template_predi_index:
                self.template_predi.append(all_predicates[i])
            print("⭐️ The indexes are:")
            print(self.template_predi_index)
            print("⭐️ Selected predicate according to the template are")
            print(self.template_predi)
            self.w = tf.Variable(
                initial_value=tf.random.truncated_normal(shape=(self.rules, len(self.template_predi_index), self.addtional_aux_num), dtype="float32"),
                trainable=True,
                constraint=lambda t: tf.clip_by_value(t, 0, 1)
            )
        else: 
            self.w = tf.Variable(
                initial_value=tf.random.truncated_normal(shape=(self.rules, len(self.template_predi_index), self.addtional_aux_num), dtype="float32"),
                trainable=True,
                constraint=lambda t: tf.clip_by_value(t, 0, 1)
            )
        return 0

    
    def spatial_unity(self, x):
        '''
        Define the sptial unity:
        - 1 BASIC CONSTRAIN. Ensure that the variable x and y appear in the body. 
            - Update: Ensure the variable in the head apper in the body 
        - 2 OCCURANCE CONSTRAIN. Ensure the occurrence of variable z in the body cannnot be one 
            - Update: Ensure the rest variable not  in the head connot be one. 
            - The index in the embedding of all predicate are fixed x:0 y:1 z:2
        '''
        # Do multiple between parameters and its embeddings. The first arity         
        weights = tf.reduce_mean(self.w, axis=2) 
        weights = tf.expand_dims(weights,1)
        weighted_embeddings = weights * self.e_o_p
        constrain_x_and_y = weighted_embeddings[:,0:self.arity,:] # keep the first arity embeddings 
        constrain_rest = weighted_embeddings[:,self.arity:,:] # The latter embeddigns 
        neg_weighted_embedddings = 1-constrain_x_and_y
        reduce_pro = tf.math.reduce_prod(neg_weighted_embedddings, 2)
        fuzzy_or =  1 - reduce_pro # This result is the fuzzy or between all predicate for same boolean variable and
        print(fuzzy_or)
        fuzzy_and = tf.math.reduce_prod(fuzzy_or, 1) # This result is the fuzzy and between all Boolean variable in the all rules 
        print(fuzzy_and)
        fuzzy_and = tf.reshape(fuzzy_and, [-1, self.rules])
        # all_rule_fuzzy_and = tf.math.reduce_prod(fuzzy_and, 0)
        # all_rule_fuzzy_and = tf.reshape(all_rule_fuzzy_and, [-1,1])
        print("[⭐️ np]: The final of sptial output")
        print(fuzzy_and)
        sptial_output = tf.matmul(x, fuzzy_and, name='sptial_utility') 

        # The second arity 
        the_sum_all_rest = tf.math.reduce_sum(constrain_rest,2)
        sptial_activate = self.sptial_activation(the_sum_all_rest)
        sum_sptial_activate = tf.math.reduce_sum(sptial_activate, 1)
        sptial_activate_transpose = tf.reshape(sum_sptial_activate, [-1, self.rules])
        sptial_output_2 = tf.matmul(x, sptial_activate_transpose, name='sptial_utility_rest_variable')
        
        return sptial_output, sptial_output_2
    
    def max_similarity(self, list_weights, one_input):
        '''
        In the keras API, the similarity = 1, then the vector is different 
        '''
        shape_weight = list_weights.get_shape().as_list()
        number_AB_rule = shape_weight[0]
        number_AH_rule = shape_weight[2]
        all_combination_rows = list(itertools.combinations(range(number_AH_rule),2))
        cosine_loss = tf.keras.losses.CosineSimilarity(axis=1)
        dissimi = []
        for i in range(number_AB_rule):
            for j in all_combination_rows:
                y_1 = tf.reshape(list_weights[i,:,j[0]],[1,-1])
                y_2 = tf.reshape(list_weights[i,:,j[1]],[1,-1])
                diss = tf.reshape(tf.convert_to_tensor(cosine_loss(y_1, y_2)), [1])
                dissimi.append(diss)
        dissmi_tf = tf.concat(dissimi,axis=0)
        # y_1 = all_combination_rows[:,0]
        # y_2 = all_combination_rows[:,1]
        # cosine_loss = tf.keras.losses.CosineSimilarity(axis=1)
        dissimilarity = tf.reshape( dissmi_tf,[1,-1])
        final_dis = tf.matmul(one_input, dissimilarity)
        return final_dis
    
    
    def DLFIT_inference(self, x):
        template_predicate = tf.gather(x, self.template_predi_index, axis=1, name='aux_inf')
        average_sum_aux = tf.reduce_mean(self.w, axis=2)
        average_sum_aux_reshape = tf.reshape(average_sum_aux, [-1,self.rules])
        fuzzy_or_value = self.sigmoid_like(tf.matmul(template_predicate, average_sum_aux_reshape)  -1)
        fuzzy_or_value = super(auxiliary_inference, self).fuzzy_or(fuzzy_or_value)
        return fuzzy_or_value

    
    def call(self, inputs,input_all_one):
        # The inference computation, the sptial computation and the one sum computation 
        logic_inference = self.DLFIT_inference(inputs)
        basic_constrains, occurance_constrains  = self.spatial_unity(input_all_one)
    
        # Make the sum of all weights in a single rule be one.
        weights = tf.reduce_mean(self.w, axis = 2)    
        sum = tf.reduce_sum(weights,axis=1)
        sum_tem = tf.reshape(sum, [-1,self.rules])
            
        #print(sum_tem.shape)
        one_sum_constrains = tf.matmul(input_all_one, sum_tem, name='one_sum')    
        #print("[⭐️ np]: Shape of one_sum",one_sum_constrains)
        
        # TODO Compute the dissimilarity between all rows, we will return the meaned sum of all value, we can call the keras loss API 
        # TODO The similiarity should be 1  (most dissimilar)
        similiarity = self.max_similarity(self.w, input_all_one) # should be 1 in shape[None, 5]
        
        
        # TODO ONE_SUM_CONSTRAIN and BASIC_CONSTRAIN should eauql with one, OCCURSNCE_CONSTRAIN shoule be zeros
        return [logic_inference, one_sum_constrains, basic_constrains, occurance_constrains, similiarity]
    
    
    def get_config(self):
        return {"inference_weights": self.w.numpy(), "sptial_embeddings":self.e_o_p.numpy()}

    @classmethod
    def from_config(cls, config):
        return cls(**config)

    def return_template_predi_index(self):
        return self.template_predi_index

class make_prediction_using_logic_program(keras.layers.Layer):
    def __init__(self, logic_program_dic ):
        super(make_prediction_using_logic_program, self).__init__()
        self.logic_program_dic = logic_program_dic
        
    def call(self):
        return 0


In [32]:
# this code defines multilabel neural predicates
# -*- coding: utf-8 -*-
'''
set variable depth as k, and set all variables V in the datasets
V = 2 + k (2 indicates the number of two prime variables x and y)


set the input of neural networks, including two embeddings represetning two objects
set A(|V|, 2) + |V| as the number of output nodes of neural predicates

We regard multilabel neural networks as the neural predicate
'''
import os
import pickle
from re import A
import sys
import matplotlib.pyplot as plt 
from pathlib import Path

from numpy.core.records import array


#path = os.getcwczd()

path = Path(os.getcwd())
parent_path = path.parent.absolute()
print('Parent path is:')
print(parent_path)
sys.path.append(os.getcwd())
sys.path.append(str(parent_path))
print(sys.path)
print("----")
#sys.path.append(os.getcwd())
import logging
logger = logging.getLogger()
logger.basicConfig = logging.basicConfig(level=logging.DEBUG)
import tensorflow as tf
import tensorflow.keras as keras 
import tensorflow.keras.backend as K
import numpy as np
# import DLFIT as DLFIT
import argparse
import itertools


# There is only one neural predicate in the current implementation

# set the number of layers in each neural predicate

LAYERS = 4 #default value 
CONSTANTS = {} # I think this is useless untill we do some constrains on the embeddings of the objects in the task 
PREDICATE = {} # The PREDICATE dictionary record all output from neural predicates 
OUTPUT = {} # The OUTOUT dictionary record all output layer from the both neural predicates and neural logic programs

# set trainable embeddings for variables
# def constant(label,value=None, min_value=None, max_value=None):
#     label = "ltn_constant_"+label
#     if value is not None:
#         result = tf.constant(value,name=label) 
#     else:
#         result = tf.Variable(tf.random.uniform(
#                 shape=(1,len(min_value)),
#                 minval=min_value,
#                 maxval=max_value,name=label))
#     result.doms = []
#     CONSTANTS[label] = result
#     return result


# def variable(label,number_of_features_or_feed):
#     if type(number_of_features_or_feed) is int:
#         result = tf.placeholder(dtype=tf.float32,shape=(None,number_of_features_or_feed),name=label)
#     elif isinstance(number_of_features_or_feed,tf.Tensor):
#         result = tf.identity(number_of_features_or_feed,name=label)
#     else:
#         result = tf.constant(number_of_features_or_feed,name=label)
#     result.doms = [label]
#     return result

def input_model(number_all_predicate):
    global input_layer 
    global input_2 
    global model_input
    input_layer = tf.keras.layers.Input(shape = (number_all_predicate,))
    input_2 = keras.layers.Input(shape = (1,))
    model_input = []
    model_input.append(input_layer)
    model_input.append(input_2)
    print("🌠 The input for pure neural logic programs build successfully!")
    return 0 
    
# set embedding layers for the objects
def define_embedding( single_input_length, embedding_size, number_arity = 2, variable_depth = 1):
    global flatten_layer 
    global input_layer
    global number_of_variable
    global input_2
    global model_input
    number_of_variable = number_arity + variable_depth

    # Define the input of the model
    input_layer = tf.keras.layers.Input(shape= (number_of_variable,), name='Input_Layer_pred')
    input_2 = keras.layers.Input(shape=(1,))
    # Assemble all input layer 
    model_input = []
    model_input.append(input_layer)
    model_input.append(input_2)
    embedding_layer = tf.keras.layers.Embedding(input_dim = single_input_length, output_dim = embedding_size, input_length=number_of_variable)(input_layer)
    # Connect the constants and their embeddings at here...
    
    flatten_layer = tf.keras.layers.Flatten()(embedding_layer)
    print("[⭐️ nID⭐️] Construct embedding layer successfully")
    return 0

#set binary neural predicates 
def binary_predicate(label='', t_relation=False, n_layer=LAYERS, pred_definition=None,):
    global flatten_layer
    number_of_predicate = len(list(itertools.permutations(range(number_of_variable),2))) # no unary predicates
    if t_relation == True:
        number_of_predicate = number_of_predicate - 1
    
    print('[⭐️ nID⭐️] The number of predicates for [',label,'] is',number_of_predicate) 
    tem_hidden_layer = tf.keras.layers.Dense(20, activation = 'relu', name = label+'_HL_'+str(0))(flatten_layer)
    if pred_definition is None:
        # define layer and weight parameters
        for i in range(n_layer-1):
            tem_hidden_layer = tf.keras.layers.Dense(20, activation = 'relu', name = label+'_HL_'+str(i+1))(tem_hidden_layer)
        output_layer = tf.keras.layers.Dense(number_of_predicate, activation = 'sigmoid', name = label+'_OL')(tem_hidden_layer)
    else: 
        print("[⭐️ nID⭐️] The predicate has been defined, and it cannot be defined again!")
        return 
    
    PREDICATE[label] = output_layer 
    OUTPUT[label] = output_layer 
    print('[⭐️ nID⭐️] Construct neural predicate for [',label,'] successfully') 
    return 0

# Define the neural logic programs with same head fitsr-order features. Like the RNN model, bulid a recurrent between input and output
def neural_logic_program(label='', n_rules=4, alpha = 10, predicate_path='', 
                         template = None, target_index=None, 
                         only_logic_program = False, target_arity = 2, 
                         prior_knowledge = False, variable_number = 3):
    global logic_program
    p_all_r = []
    if only_logic_program == False: 
        # Combine the values from all predicates into one vector in shape [|number of all predicates|,]
        for i in PREDICATE:
            p_all_r.append(PREDICATE[i])
        p_all_r = tf.concat(p_all_r, axis=1)
    else:
        p_all_r = input_layer
            
    logic_program = inference(n_rules=n_rules, alpha=alpha, data_path=predicate_path, path=predicate_path+'predicate.list', 
                                    template=template, target_index = target_index, 
                                    arity=target_arity, l_name= 'Original_layer',
                                    prior_knowledge = prior_knowledge,
                                    variable_number = variable_number) # define a logic_program obejct
    
    # Compute the infernece layer 
    logic_program_output = logic_program(p_all_r, fun_index = 1)
    inference_layer = logic_program_output['inference_output']
    final_predicate = logic_program.fuzzy_or(inference_layer) # inference layer 
    OUTPUT[label] = final_predicate
    
    if target_index != None and only_logic_program == False:
        predicted_output = logic_program.target_predicate_predited_neural(p_all_r)
        OUTPUT[label+'directly_pred'] = predicted_output
    # Compute the one loss layer
    #logic_program_output_2 = logic_program.one_sum(input_2)
    #one_loss_sum = logic_program_output_2
    one_sum_output = logic_program(input_2, fun_index = 2)
    OUTPUT[label+'one_loss'] = one_sum_output['one_sum_loss'] # ❓ A output layer for custimized loss , a float number 
    
    #sptial_output = logic_program.spatial_unity() # Add sptial constrains
    #OUTPUT[label+'sptial'] = sptial_output 
    
    sptial_output, sptial_output_2 = logic_program.spatial_unity(input_2)
    OUTPUT[label + 'basic'] = sptial_output
    OUTPUT[label + 'occurance'] = sptial_output_2
    
    if prior_knowledge != False:
        OUTPUT[label+'prior_cos'] = logic_program(input_2, fun_index = 3)['cos_prior']
    
    
    # TODO Tis code is added in the recernt version 
    auxiliary_layer = auxiliary_inference( n_rules = n_rules,alpha = alpha,
                                                data_path = predicate_path, path=predicate_path+'predicate.list',
                                                template=template, target_index = target_index,arity=target_arity, 
                                                l_name= 'AUX_layer', addtional_aux_num=2,
                                                aux_prior_knowledge = prior_knowledge, variable_number= variable_number)
    aux_res = auxiliary_layer(inputs = p_all_r,input_all_one= input_2)
    OUTPUT['aux_inf'] = aux_res[0]
    OUTPUT['aux_one_sum'] = aux_res[1]
    OUTPUT['aux_basic_loss'] = aux_res[2]
    OUTPUT['aux_occurance_loss'] = aux_res[3]
    OUTPUT['aux_dissimiliarity_loss'] = aux_res[4]
    return 0 

def build_model(figure_path = None):
    global model_output
    global model 
    # Define the model 


    print("[⭐️ nID ⭐️] All output layer is")
    for i in OUTPUT:
        print(OUTPUT[i])

    # try the ombination output
    #new_output = tf.concat(list(OUTPUT.values()), axis= 1)
    
    
    model_output = OUTPUT.values() # A list includes all the output layers which we have defined before 
    model = tf.keras.Model(inputs = model_input,  outputs = model_output)
        
    # Print the model summary
    print("[⭐️ nID ⭐️] Model summary is:")
    print(model.summary())
    
    # print the details of the model
    # print('[⭐️ nID⭐️] Print model...')
    # tf.keras.utils.plot_model(model, to_file=figure_path+'/model_plot.png', show_shapes=True, show_layer_names=True)
    # print('[⭐️ nID⭐️] Print the details of model success!')
    

    return 0
    
# set trainable function to train the neural predcates in the PREDICATE dictionary
def train(x, y, max_epochs=10000, learning_rate= 0.02, decay_rate= 1, model_path = None ,verbose = 1, patience = 100, figure_path = None, batch_size = 32, validation_split = 0.2 ,loss_weight=None, relation_number = None, prior_knowledge = False, validation_data = None):
    
    # print('All constants include',CONSTANTS) # we do not need constant layer when we already set the Embedding layer.    
    # Define the custom loss function 
    # DLFIT_one_sum_loss = DLFIT.inference.loss_sum_one # ? Fix at here
    

    # Define the metrics and training algorithms
    print("[⭐️ nID⭐️] Define the loss and metrics...")
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=learning_rate,decay_steps=10000,decay_rate=decay_rate) # it is not decay now
    opt = tf.keras.optimizers.Adam(learning_rate=lr_schedule) # Using Adam as the optimization function. We can try SGD later.
    metrics_ce = []
    #metrics_ce.append(tf.keras.metrics.Precision())
    #metrics_ce.append(tf.keras.metrics.Recall())
    metrics_ce.append(tf.keras.metrics.BinaryAccuracy())
    # Define the loss list 
    loss_list = []
    #loss_list.append("mean_squared_error")
    loss_list.append('binary_crossentropy')  # Let predicate of predicate and target predicad are trained thourgh CrossEntropy  
    loss_list.append('mean_squared_error') # Let one_sum and sptical loss are trained through MSE 
    loss_list.append('mean_squared_error') 
    loss_list.append('mean_squared_error') 
    if prior_knowledge != False:
        loss_list.append('mean_squared_error')
    # TODO For Auxiliary Predicate 
    loss_list.append('binary_crossentropy')  # Let predicate of predicate and target predicad are trained thourgh CrossEntropy  
    loss_list.append('mean_squared_error') # Let one_sum and sptical loss are trained through MSE 
    loss_list.append('mean_squared_error') 
    loss_list.append('mean_squared_error') 
    loss_list.append('mean_squared_error') 

    model.compile(loss=loss_list, optimizer=opt,  metrics=metrics_ce,loss_weights=loss_weight) # ⭐️ check whether is right. I guess the threshhold in the metrics computing function is 0.5
    

    
    # Define the callback functions for 
    es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=patience)
    
    # cir_learning = CurriculumLearning(logic_program, validation_data, relation_number)
    
    # combine y 
    # y = np.concatenate(y, axis=1)
    print("[⭐️ nID⭐️] Fit model on training data:")
    
    history = model.fit(
        x,
        y,
        batch_size=batch_size,
        epochs=max_epochs,
        validation_split = validation_split,
        verbose=verbose,
        callbacks=[es],
        validation_data=validation_data,
    )
    
    attributes = list(history.history.keys())
    print('[⭐️ nID⭐️] The attributes in the model include',attributes)
    
    # Save the model
    if model_path != None:
        model.save(model_path)
        print("[⭐️ nID⭐️] Save model successfully")
    
    # Polt the metrics in the model
    if figure_path != None:
        plot_metrics(attributes, history, figure_path)
        
    return 0
    
def plot_metrics(attributes, history, figure_path):
    
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    
    plt.title('loss of all NLP')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['loss','val_loss'], loc='right')
    plt.savefig(figure_path+'/loss.png')
    plt.clf()   
    print("[⭐️ nID⭐️] Plot metrices successfully! ")
    return 0

def test_model(x, model_path = None):
    if model_path != None:
        model = keras.models.load_model(model_path)
    print(len(list(model.layers)))
    for i in range(1, len(model.layers)):
        layer_output = tf.keras.backend.function([model.layers[0].input], [model.layers[i].output])
        layer_output = layer_output(x)
        print("[⭐️ nID⭐️] Test output from the",i,'th is',layer_output)
    return 0 
    
def prediction(model, true_value_dictionary):
    # with open(data_path+'/valid_index.dt','rb') as f:
    #     valid_index = pickle.load(f)
    #     f.close()
    # v_index = valid_index['valid_index']
    # template = valid_index['template']
    output = model.predict(true_value_dictionary)
    print("[⭐️ nID⭐️] The prediction is:")
    print(output)
    return output

def extract_symbolic_logic_programs(relation = [], data_path = '', 
                                    result_path = '', model = None, threshold = 0.0,
                                    head_pre = '', number_of_variable = 3):
    with open(data_path+'/valid_index.dt','rb') as f:
        valid_index = pickle.load(f)
        f.close()
    with open(data_path+'/relation_variable.dt','rb') as f:
        relation_variable_permutations = pickle.load(f)
        f.close()
    v_index = valid_index['valid_index']
    template = valid_index['template']
    reconstructed_model = model
    
    print("[⭐️ nID⭐️ ] The number of all layer is:") 
    print(len(reconstructed_model.layers))
    
    layer = reconstructed_model.get_layer(name = 'Original_layer')
    all_layer_in_original = layer.get_weights()
    layer_weights_o = all_layer_in_original[0] # The first element in this list is the trainable 
    
    layer_aux_ini = reconstructed_model.get_layer(name = 'AUX_layer')
    all_layer_aux = layer_aux_ini.get_weights()
    layer_aux = all_layer_aux[0]
    layer_aux_mean_sum = np.mean(layer_aux,axis=2)
    layer_aux_final = np.transpose(layer_aux_mean_sum)
    
    
    layer_weights = np.concatenate((layer_weights_o, layer_aux_final) , axis=1)
    
    print(layer_weights)
    
    # set the variable in the dic to X Y Z and additional variable to W1 W2 W3...
    dic = {0:'X', 1:'Y', 2:'Z'}
    basic_number = 3
    additional_variable_name_pool = ['W','M','N','T']
    for i in range(3,number_of_variable):
        dic[i] = additional_variable_name_pool[i-basic_number]
    
    # # save the variable dic
    # with open(data_path+'/variableinfo.txt', 'w') as f:
    #     print(dic.keys(), file = f)
    #     f.close()
    # with open(data_path+'/variableinfo.dt', 'wb') as f:
    #     pickle.dump(dic.keys(), file = f)
    #     f.close()

    target_predicate_tuple=[]
    # According to the index, reshow the symbolic information stored in the following list
    all_variable_symbolic = []
    for item in relation_variable_permutations:
        head_symbol = item
        body_list = relation_variable_permutations[item]
        for pair_variable in body_list:
            first_variable = dic[pair_variable[0]]
            second_variable = dic[pair_variable[1]]
            latter = '('+first_variable+','+second_variable+')'
            target_predicate_tuple.append((head_symbol,latter))
    
    
    
    target_predicate = []
    for item in range(len(target_predicate_tuple)):
        if item in v_index:
            target_predicate.append(target_predicate_tuple[item][0]+target_predicate_tuple[item][1])
        

    #extract the symbolic logic format 
    max_value_tensor = np.amax(layer_weights)
    if max_value_tensor < threshold:
        print("No acceptable elements in the training dataset according to the threshold.")
        return -1, layer_weights
    statisfied_index = np.transpose((layer_weights >= threshold).nonzero())
    # print(statisfied_index)
    satisfied_list = []
    for i in statisfied_index:
        one = []
        one.append(i[1])
        one.append(i[0])
        satisfied_list.append(one)
    
    satisfied_list = sorted(satisfied_list, key=lambda tup: tup[0])    
    # print(satisfied_list)
    
    try:
        number_of_rule = satisfied_list[-1][0] + 1 
    except IndexError:
        return -2,-2 
    information_logic = {}
    
    for i in range(number_of_rule):
        information_logic[i] = []
    
    for i in satisfied_list:
        f_i = i[0]
        s_i = i[1]
        information_logic[f_i].append(s_i)
    print("[⭐️ nID ⭐️] predicate index after the logic index")
    print(information_logic)
    rules = []
    for i in information_logic:
        rule_one = head_pre + ' :- '
        for item in information_logic[i]:
            rule_one += target_predicate[item] + '& '
        rules.append(rule_one)
    # print(rules)
    
    return rules, layer_weights


class CurriculumLearning(keras.callbacks.Callback):
    """Stop training when the precision and recall of the logic rules are both one. 
    The sumbolic methods (algorithm)
    Arguments:
    The weight
    """

    def __init__(self,logic_program_c,validation_data, relation_number):
        super(CurriculumLearning, self).__init__()
        self.logic_program = logic_program_c
        self.validation_data = validation_data
        self.template_predicate_index = None
        self.relation_number = relation_number

    def on_train_begin(self, logs=None):
        self.presicion_rule = 0
        self.recall_rule = 0
        # The epoch the training stops at.
        self.stopped_epoch = 0
        self.template_predicate_index = self.logic_program.return_template_predi_index()
        print("The template indexs are:")
        print(self.template_predicate_index)


    def on_epoch_begin(self, epoch, logs=None): # change to the end
        # Show all validation 
        print("Validation data")
        print(self.validation_data)
        val_data = self.validation_data[1]
        empty_one = []
        for i in range(self.relation_number):
            empty_one.append(val_data[i])
        val_data = tf.concat(empty_one, axis = 1)
        val_label = self.validation_data[1][self.relation_number]
        print(val_data, val_label)
        # get the label of the validation data in the corresponding indexes
        if self.template_predicate_index != None:
            valid_predicate_data = tf.gather(val_data, self.template_predicate_index, axis=1)
            print("Valid_data")
            print(valid_predicate_data)
        else:
            print("Set logic template:")
            
        # Extract the logic program using the defined method
        layer = model.get_layer(index = -6)
        layer_weights = layer.get_weights()  
        print(layer_weights)
        
        
        
        # Make predictin in the defined method in the DLFIT class or redefined ?
        
        # Compute the predision and recall of the generated symboloc program 
                
        # Begin to extract the logic program from the learned matrices 
        # rules= extract_symbolic_logic_programs(relation=['locatedIn','neigibor'],head_pre='locatedIn(x,y)',
        #                                         dataset = data_path, model = '/result/model',
        #                                         number_of_variable=3, threshold = 0.)
        # print("::> All rules extracted from neural logic prgoram are:")
        # for i in rules:
        #     print(i)
        
        
        # current = logs.get("loss")
        # if np.less(current, self.best):
        #     self.best = current
        #     self.wait = 0
        #     # Record the best weights if current results is better (less).
        #     self.best_weights = self.model.get_weights()
        # else:
        #     self.wait += 1
        #     if self.wait >= self.patience:
        #         self.stopped_epoch = epoch
        #         self.model.stop_training = True
        #         print("Restoring model weights from the end of the best epoch.")
        #         self.model.set_weights(self.best_weights)

    def on_train_end(self, logs=None):
        if self.stopped_epoch > 0:
            print("Epoch %05d: early stopping" % (self.stopped_epoch + 1))


Parent path is:
/Users/imansharifi/Documents/VirtualEnvs/MachineLearning/Code/Literature
['/Users/imansharifi/.pyenv/versions/3.12.8/lib/python312.zip', '/Users/imansharifi/.pyenv/versions/3.12.8/lib/python3.12', '/Users/imansharifi/.pyenv/versions/3.12.8/lib/python3.12/lib-dynload', '', '/Users/imansharifi/.pyenv/versions/3.12.8/lib/python3.12/site-packages', '/Users/imansharifi/Documents/VirtualEnvs/MachineLearning/Code/Literature/DFOL', '/Users/imansharifi/Documents/VirtualEnvs/MachineLearning/Code/Literature', '/Users/imansharifi/Documents/VirtualEnvs/MachineLearning/Code/Literature/DFOL/KnowledgeGraphEmbedding/codes', '/Users/imansharifi/Documents/VirtualEnvs/MachineLearning/Code/Literature/DFOL', '/Users/imansharifi/Documents/VirtualEnvs/MachineLearning/Code/Literature', '/Users/imansharifi/Documents/VirtualEnvs/MachineLearning/Code/Literature/DFOL', '/Users/imansharifi/Documents/VirtualEnvs/MachineLearning/Code/Literature', '/Users/imansharifi/Documents/VirtualEnvs/MachineLearni

/var/folders/y3/py52bzvn67nbjv67d4vr2ns80000gp/T/ipykernel_1958/3324764943.py:20: DeprecationWarning: numpy.core.records is deprecated and has been renamed to numpy._core.records. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.records.array.
  from numpy.core.records import array


In [33]:
# -*- coding: utf-8 -*-
import os
from re import M
import sys
import numpy as np
from pathlib import Path

from tensorflow.python.keras.backend import dtype, variable

#path = os.getcwczd()
path = Path(os.getcwd())
parent_path = path.parent.absolute()
sys.path.append(os.getcwd())
sys.path.append(str(parent_path))
#sys.path.append(os.getcwd())
import logging
logger = logging.getLogger()
logger.basicConfig = logging.basicConfig(level=logging.DEBUG)
import nID as nID
import tensorflow as tf
import numpy as np
import argparse
import itertools
import pickle
import time

def _parse_function(example_proto):
    feature =  {'body': tf.io.FixedLenFeature([], tf.string), 
            'head':tf.io.FixedLenFeature([], tf.string)
            }
    example = tf.io.parse_single_example(example_proto, feature)
    example_body = example['body']
    example_body = tf.io.parse_tensor(example_body, tf.double)
    # example_body = tf.reshape(example_body,[1,-1])
    
    example_head = example['head']
    example_head = tf.io.parse_tensor(example_head, tf.double)
    example_head = tf.reshape(example_head, [-1,])

    one = tf.constant([1.0])
    zero = tf.constant([0.0])
    example_body = (example_body, one)
    if prior != False:
        example_head = (example_head, one, one, zero, zero, example_head, one, one, zero, one )
    else: 
        example_head = (example_head, one, one, zero, example_head, one, one, zero, one)
    
    return example_body, example_head

def t_main(dataset_dir, t_relation, alpha=10, learning_rate=0.001, target_arity = 2, max_epochs = 1000, n_rules = 4, prior_knowledge = False, variable_depth = 1, larger = False, batch_size = 32, verbose = 2):

    
    n_rules = n_rules # The number of rules in the nerual logic programs

    task_data_path = dataset_dir
    
    folder_name = dataset_dir
    data_folder = dataset_dir

    variable_arity = 2
    variable_number = variable_depth + variable_arity
    
    # Open entity files 
    with open(data_folder+'all_ent.dt', 'rb') as f:
        entities = pickle.load(f)
        f.close()
    
    print("👉 The number of all entities is", len(entities))
    global prior
    with open(data_folder+'all_relation_dic.dt','rb') as f:
        relations = pickle.load(f)
        f.close()
    
    try:
        with open(data_folder+'prior_knowledge.dt', 'rb') as f:
            prior = pickle.load(f) 
            flag = not np.any(prior)
            if flag:
                print('prior_knowledge is empty')
                prior_knowledge = False
            else:
                print('prior_knowledge is not empty')
            f.close()
    except FileNotFoundError:
        prior_knowledge = False
        
    prior = prior_knowledge
    target_index = 0 # the index is 0 or 6 
    
    entities_dic = {}
    index_dic = 0 
    for i in entities:
        entities_dic[index_dic] = i 
        index_dic += 1
    print("The dictionary about entities",entities_dic)
    
    
    if larger == True:
        dataset = tf.data.TFRecordDataset(task_data_path+'data.tfr')
        dataset = dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE)
        
        for i in dataset.take(1):
            body_sample = i[0]
            number_all_predicate = int(tf.shape(body_sample[0])[0].numpy())
            print(i[0]) # [[1*13],[1*1]]
            print(i[1]) # [[1*1]...[1*1]]

        with open(task_data_path+t_relation+'number_valid_substitution.txt') as f:
            total_number = f.readline()
            total_number = int(total_number)
            f.close()
        train_number = int(0.9 * total_number)
        
        # For true randomness, we set the shuffle buffer to the full dataset size.
        # dataset = dataset.cache() # Add from smaller data. (Less than 1GB)
        dataset = dataset.shuffle(2048) # I think the buffer size can not be the whole dataset. 
        
        train_data = dataset.take(train_number)
        valid_data = dataset.skip(train_number)
        
        #  Batch after shuffling to get unique batches at each epoch.
        train_data = train_data.batch(batch_size) 
        train_data = train_data.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
        
        valid_data = valid_data.batch(batch_size)
        valid_data = valid_data.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
        
        #for i in train_data:
        #    print(i[0])
        #    print(i[1])
    
    else:

        # extract the datasaets
        x_file = task_data_path+'x_train.dt' 
        y_file = task_data_path+'y_train.dt'

        
        with open(x_file, 'rb') as xf:
            x = pickle.load(xf)
            xf.close()
        with open(y_file, 'rb') as yf:
            label = pickle.load(yf)
            yf.close()
            
        # Combine all predicates in all relations 
        x = np.concatenate(x,axis=1)
        #  x_test = np.concatenate(x_test,axis=1)

        number_all_predicate = int(np.shape(x)[1])
        
        new_x = []
        # new_x_test = []
        
        new_x.append(x)
        # new_x_test.append(x_test)
        
        y = []
        # y_test = []

        y.append(np.array(label))
        # y_test.append(np.array((label_test)))
            
        #Make more data for sptial constrains and one_sum constrains
        shape_data = len(label) # Get the first shape of the training data

        y.append(np.ones((int(shape_data), 1),dtype= float))
        y.append(np.ones((int(shape_data), 1),dtype = float))  # Make label for the final label data
        y.append(np.zeros((int(shape_data), 1), dtype= float))
        if prior_knowledge != False:
            y.append(np.zeros((int(shape_data), 1 ), dtype= float)) # We do not know the explicit number of rows, hence we use 1 
        
        #TODO for aux 
        y.append(np.array(label))
        y.append(np.ones((int(shape_data), 1),dtype= float))
        y.append(np.ones((int(shape_data), 1),dtype = float))  # Make label for the final label data
        y.append(np.zeros((int(shape_data), 1), dtype= float))
        y.append(np.ones((int(shape_data), 1), dtype= float))
        # ? Total 9 y 
        
        new_x.append(np.ones((int(shape_data), 1 ),dtype= float)) 
        
        
        label = y 
        x = new_x
        
        
        print('::> Length of input x',len(x), 'length of label is', len(label))


    # Define neural logic program according to the DLFIT
    print("🔧 Begin to define neural logic programs")
    # define the input layer
    input_model(number_all_predicate=number_all_predicate)
    
    
    with open(data_folder+'valid_index.dt','rb') as f:
        res = pickle.load(f)
        f.close()
    template = res['valid_index']   # The valid index, inclduing the target predicate 
    
    neural_logic_program(label = t_relation, n_rules = n_rules, alpha = alpha, predicate_path=data_folder, template=template,target_index=target_index, only_logic_program=True, target_arity = target_arity, prior_knowledge = prior_knowledge, variable_number = variable_number)
    print("::> Define logic programs successfully. All output layers are",nID.OUTPUT)
    print()
    
    # Define model 
    print("::> Define the model")
    build_model(figure_path=folder_name)
    print("::> Build model scuuess!")
    print()
    
    # Train the KB
    print("::> Begin to tain the defined multilabeled model respresenting neural predicates")

    if larger == True:
        nID.train(train_data,None,max_epochs = max_epochs, learning_rate = learning_rate, batch_size = None, model_path=folder_name + '/model', figure_path=folder_name, verbose=verbose, patience=2, validation_split=0.0, loss_weight = [0.2,0.2,0.2,1,0.2,0.2,0.2,1], relation_number = len(relations), prior_knowledge = prior_knowledge, validation_data = valid_data)
    else:
        nID.train(x,label,max_epochs = max_epochs, learning_rate = learning_rate, batch_size = batch_size,model_path=folder_name + '/model', figure_path=folder_name, verbose=verbose, patience=2, validation_split=0.2, loss_weight = [0.2,0.2,0.2,1,0.2,0.2,0.2,1], relation_number = len(relations), prior_knowledge = prior_knowledge)
    
    return 0


In [34]:
def curriculum_learning(dataset_dir, dataset, predicate, learning_rate=0.001, alpha = 10, variable_depth = 1, lar = False, bs = 64, verbose = 2, final_threshold = 0.3, sample_walk = False, learning_times = 5):
    ini_alpha = alpha
    time_train = 0
    while time_train < learning_times:
        with open(dataset_dir+"meta_info.dt",'rb') as f:
            meta_info = pickle.load(f)
            f.close()
        target_arity = meta_info["target_arity"]
        # Train the mdoel 
        t_main(dataset_dir, t_relation=predicate, alpha = ini_alpha, learning_rate= learning_rate, target_arity = target_arity, max_epochs=2000, n_rules=5, prior_knowledge= True, variable_depth=variable_depth, larger = lar, batch_size = bs, verbose = verbose)
        # last learning rate 0.001
        # Do extract and save the best parameters 
        head_pre = meta_info["head_pre"]
        target_arity = meta_info["target_arity"]
        acc = get_best_logic_programs(dataset,predicate,head_pre,target_arity,variable_depth, final_threshold=final_threshold, sample_walk = sample_walk)
        if acc == -2:
            ini_alpha = int(ini_alpha/1.2)
            continue
        if acc >= 0.997:
            break
        time_train+=1
    return acc

In [35]:
dataset_dir = 'test_dir/'
dataset = 'nations.nl'
predicate = 'blockpositionindex'
learning_rate = 0.001
alpha = 10
variable_depth = 1
lar = False
bs = 64
verbose = 2
final_threshold = 0.3
sample_walk_check = False
curriculum_learning(dataset_dir, dataset, predicate, learning_rate, alpha ,variable_depth, lar, bs, verbose, final_threshold, sample_walk_check)

👉 The number of all entities is 14
The dictionary about entities {0: 'uk', 1: 'ussr', 2: 'brazil', 3: 'usa', 4: 'netherlands', 5: 'jordan', 6: 'egypt', 7: 'india', 8: 'israel', 9: 'china', 10: 'poland', 11: 'indonesia', 12: 'burma', 13: 'cuba'}
::> Length of input x 2 length of label is 9
🔧 Begin to define neural logic programs
🌠 The input for pure neural logic programs build successfully!
[⭐️ np] All predicates in the task are: [(0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 3), (3, 0), (3, 1), (3, 2), (0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 3), (3, 0), (3, 1), (3, 2), (0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 3), (3, 0), (3, 1), (3, 2), (0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 3), (3, 0), (3, 1), (3, 2), (0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 3), (3, 0), (3, 1), (3, 2), (0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 3), (3, 0), (3, 1)

ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


In [1]:
import numpy as np

def compute_mrr_hits(predicted_scores, correct_index, hits_k=(1, 3, 10)):
    """
    Compute reciprocal rank and hits@K.
    predicted_scores: np.array of score values (higher means more likely).
    correct_index: integer index of the correct item in this array.
    hits_k: tuple of k values for Hits@K.
    Returns: mrr, {k: hits@k}
    """
    # Sort in descending order, so highest score comes first
    sorted_indices = np.argsort(-predicted_scores)
    rank = np.where(sorted_indices == correct_index)[0][0] + 1  # Ranks start at 1
    mrr = 1.0 / rank
    hits = {k: int(rank <= k) for k in hits_k}
    return mrr, hits

# Dummy test:
# Imagine the correct answer is item 2, and model scored as follows:
scores = np.array([0.2, 0.5, 0.8, 0.1])
correct = 2  # Index of actual
mrr, hits = compute_mrr_hits(scores, correct)
print(f"MRR: {mrr:.3f}")           # Should be 1.0 since correct is highest
print(f"Hits@1: {hits[1]}, Hits@3: {hits[3]}, Hits@10: {hits[10]}")  # Should be hits@1=1

# More challenging example: correct isn't ranked first
scores = np.array([0.9, 0.8, 0.7, 0.6])  # correct idx=2
correct = 2
mrr, hits = compute_mrr_hits(scores, correct)
print(f"MRR: {mrr:.3f}")           # Should be 1/3 since correct is 3rd
print(f"Hits@1: {hits[1]}, Hits@3: {hits[3]}, Hits@10: {hits[10]}")  # hits@3=1, hits@1=0

MRR: 1.000
Hits@1: 1, Hits@3: 1, Hits@10: 1
MRR: 0.333
Hits@1: 0, Hits@3: 1, Hits@10: 1


In [3]:
import numpy as np

# Suppose you have a batch of candidate samples for many queries
# y_test: shape [num_queries, num_candidates], entries 0 or 1
# y_pred: shape [num_queries, num_candidates], entries between 0 and 1 (model outputs)

def score_for_candidate(y_pred, y_true):
    return y_pred if y_true == 1 else 1 - y_pred

def compute_scores(y_pred, y_true):
    # Vectorized over candidate arrays
    return y_pred * y_true + (1 - y_pred) * (1 - y_true)

# For one query, for example:
y_pred = np.array([0.3, 0.8, 0.1, 0.6])  # model outputs for candidates
y_true = np.array([0, 1, 0, 0])           # true label for each candidate

scores = compute_scores(y_pred, y_true)

# Find correct candidate index (where y_true == 1)
correct_idx = np.where(y_true == 1)[0][0]

# Rank candidates by scores
sorted_idx = np.argsort(-scores)
rank = np.where(sorted_idx == correct_idx)[0][0] + 1

mrr = 1.0 / rank
hits_at_1 = 1 if rank == 1 else 0
hits_at_3 = 1 if rank <= 3 else 0

print(f"Scores: {scores}")
print(f"Rank of correct candidate: {rank}")
print(f"MRR: {mrr}, Hits@1: {hits_at_1}, Hits@3: {hits_at_3}")

Scores: [0.7 0.8 0.9 0.4]
Rank of correct candidate: 2
MRR: 0.5, Hits@1: 0, Hits@3: 1
